In [ ]:
import sys
from pathlib import Path

sys.path.append(Path(".").resolve().parent.parent.as_posix())
from pydantic import BaseModel

import src.common as cn
from ollama import chat

model: str = "qwen3.5:9b"

In [ ]:
# basic
response = chat(
    model=model, messages=[{"role": "user", "content": "what is a spaniel?"}]
)
with open("tmp.md", "w") as f:
    f.write(str(response.message.content))

In [ ]:
# streaming
stream = chat(
    model=model,
    messages=[{"role": "user", "content": "What is a spaniel?"}],
    stream=True,
)
for chunk in stream:
    content = chunk["message"]["content"]
    if content == "":
        print("THINKING...", end="\r")
    print(content, end="", flush=True)

In [ ]:
# structured output
from typing import Literal


class Dog(BaseModel):
    breed: str
    age: int
    name: str
    color: Literal["brown", "black", "liver and white", "choc"]
    is_good_boy: bool
    speed: float
    weight: float


response = chat(
    model=model,
    messages=[{"role": "user", "content": "Generate a dog character NPC"}],
    format=Dog.model_json_schema(),
)
print(response.message.content)

In [ ]:
dog = Dog.model_validate_json(str(response.message.content))
dog

In [ ]:
# images
path_image = (cn.dir_images / "dog.jpg").as_posix()
response = chat(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "What is the breed and color of the dog in the image?",
            "images": [path_image],
        }
    ],
)
print(response.message.content)